In [1]:
import numpy as np
import os

In [2]:
path = "C:/Users/hanle/Downloads/th14_mae_g_16_4/th14_mae_g_16_4"

In [ ]:
for video in os.listdir(path):
    if video.endswith(".npy"):
        data = np.load(os.path.join(path, video))
        print(data.shape)
        # break
        

(250, 1408)
(499, 1408)
(3617, 1408)
(590, 1408)
(1555, 1408)
(1091, 1408)
(1701, 1408)
(1762, 1408)
(1331, 1408)
(867, 1408)
(853, 1408)
(109, 1408)
(738, 1408)
(1652, 1408)
(643, 1408)
(1436, 1408)
(976, 1408)
(1433, 1408)
(1261, 1408)
(4037, 1408)
(737, 1408)
(1841, 1408)
(766, 1408)
(349, 1408)
(1587, 1408)
(1570, 1408)
(1318, 1408)
(961, 1408)
(1610, 1408)
(1443, 1408)
(566, 1408)
(1748, 1408)
(1812, 1408)
(1038, 1408)
(1114, 1408)
(538, 1408)
(4841, 1408)
(948, 1408)
(1665, 1408)
(2201, 1408)
(910, 1408)
(6352, 1408)
(825, 1408)
(1375, 1408)
(1552, 1408)
(317, 1408)
(1566, 1408)
(1144, 1408)
(1034, 1408)
(1530, 1408)
(1159, 1408)
(2182, 1408)
(1174, 1408)
(750, 1408)
(210, 1408)
(927, 1408)
(5322, 1408)
(542, 1408)
(1783, 1408)
(1365, 1408)
(216, 1408)
(534, 1408)
(504, 1408)
(1504, 1408)
(372, 1408)
(657, 1408)
(1229, 1408)
(247, 1408)
(417, 1408)
(3236, 1408)
(1555, 1408)
(1336, 1408)
(1546, 1408)
(2448, 1408)
(1537, 1408)
(895, 1408)
(1911, 1408)
(233, 1408)
(641, 1408)
(2583,

In [5]:
from transformers import VideoMAEImageProcessor, AutoModel, AutoConfig
import numpy as np
import torch


config = AutoConfig.from_pretrained("OpenGVLab/VideoMAEv2-Large", trust_remote_code=True)
processor = VideoMAEImageProcessor.from_pretrained("OpenGVLab/VideoMAEv2-Large")
model = AutoModel.from_pretrained('OpenGVLab/VideoMAEv2-Large', config=config, trust_remote_code=True)

c:\phd\.venv\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [8]:
video = list(np.random.rand(16, 3, 224, 224))

# B, T, C, H, W -> B, C, T, H, W
inputs = processor(video, return_tensors="pt")
inputs['pixel_values'] = inputs['pixel_values'].permute(0, 2, 1, 3, 4)

with torch.no_grad(): 
    outputs = model(**inputs)

print(outputs.shape)

torch.Size([1, 1024])


In [9]:
import numpy as np
import torch
from transformers import VideoMAEImageProcessor, AutoModel, AutoConfig

config = AutoConfig.from_pretrained("OpenGVLab/VideoMAEv2-Large", trust_remote_code=True)
processor = VideoMAEImageProcessor.from_pretrained("OpenGVLab/VideoMAEv2-Large")
model = AutoModel.from_pretrained("OpenGVLab/VideoMAEv2-Large", config=config, trust_remote_code=True)
model.eval().cuda()

WINDOW = 16   # frames per window
STRIDE = 4    # frames between windows

    
frames = np.random.rand(32, 224, 224, 3)  # for testing only, replace with actual frames


feats = []
for start in range(0, len(frames) - WINDOW + 1, STRIDE):
    clip = frames[start:start + WINDOW]                 # [16, H, W, C]
    clip = list(clip)                                     # processor expects a list of frames
    inputs = processor(clip, return_tensors="pt")
    inputs["pixel_values"] = inputs["pixel_values"].permute(0, 2, 1, 3, 4).cuda()
    with torch.no_grad():
        out = model(**inputs)
    feats.append(out.last_hidden_state.mean(dim=1).cpu().numpy())  # pool to 1 vector

output =  np.concatenate(feats, axis=0)  # [T, 1408]
print(output.shape)

c:\phd\.venv\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
'(ProtocolError('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')), '(Request ID: d49b38dd-5c29-4fc1-a450-b84bdb30b6c6)')' thrown while requesting HEAD https://huggingface.co/OpenGVLab/VideoMAEv2-Large/resolve/main/config.json
Retrying in 1s [Retry 1/5].


AttributeError: 'Tensor' object has no attribute 'last_hidden_state'